# Bản mini: ghi -> train -> chấm, chạy ở máy nhà trong ~10 phút

Đúng 3 bước như notebook Colab (`colab/so101_act_colab.ipynb`), nhưng mọi thứ thu nhỏ để **hiểu cách chạy**, không để ra mạng giỏi:

| Bước | Ở đây | Bản thật trên Colab |
|---|---|---|
| 1. Ghi dataset | 3 ván | 500 ván + 800 ván DAgger |
| 2. Train | 300 step, batch 4 | 60 000 step, batch 32 (~3,7 giờ trên L4) |
| 3. Chấm | 2-3 seed | 774 seed |

Mạng mini chỉ train 300 step nên **sẽ gắp hỏng**: đó là bình thường. Ô 3b chấm thử một checkpoint thật có sẵn ở `D:/vla` để so sánh.
Ô 4 mở "hộp đen": tự viết vòng lặp nhìn -> đoán -> làm, ~15 dòng, để thấy mạng nhận gì và trả gì.

**Chạy:** mở file này trong VS Code, chọn kernel `D:/vla/venv-lerobot/Scripts/python.exe`, chạy lần lượt từng ô (Shift+Enter).
Mọi file sinh ra nằm ở `D:/vla/mini/` (xoá thoải mái).

In [ ]:
# Ô 0: chuẩn bị. run() chạy một module Python giống ô "!python -m ..." của Colab và in output (bỏ dòng cảnh báo cho gọn).
import os, shutil, subprocess, sys
from pathlib import Path

import sim_engine                                    # gói đã cài kiểu "editable": trỏ thẳng vào thư mục sim-engine/
REPO = Path(sim_engine.__file__).parent              # .../VLA/sim-engine
MINI = Path('D:/vla/mini')                           # chỗ chứa dataset + checkpoint của bản mini
PY = sys.executable                                  # python của kernel đang chạy (venv-lerobot)

def run(module, *args, keep=lambda line: True):
    env = {**os.environ, 'PYTHONIOENCODING': 'utf-8'}
    p = subprocess.Popen([PY, '-m', module, *map(str, args)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding='utf-8', errors='replace', env=env)
    for line in p.stdout:
        noise = 'warn' in line.lower() or line.startswith(('Svt[', '[mp4', 'Map:'))   # cảnh báo, log mã hoá video, thanh tiến độ
        if not noise and keep(line):
            print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'{module} lỗi (mã {p.returncode})')

import torch
print('repo:', REPO)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có (sẽ rất chậm)')

## 1. Ghi dataset: thầy giáo (policy viết tay min-jerk) làm mẫu 3 ván (~1 phút)

`learning/record_dataset.py`: mỗi bước lấy ảnh 2 camera (`top`, `wrist`) + 6 góc khớp, hỏi thầy action, ghi lại thành một **khung**.
Chỉ giữ ván thành công. Seed bắt đầu từ 3000 (không trùng bộ chấm 0-2499).

In [ ]:
DS = MINI / 'datasets' / 'so101_mini'
shutil.rmtree(DS, ignore_errors=True)                # LeRobot không ghi đè dataset cũ
run('sim_engine.learning.record_dataset', '--episodes', 3, '--cameras', 'top,wrist',
    '--root', DS, '--repo-id', 'local/so101_mini')

### 1b. Mở dataset xem bên trong có gì

Một khung = các trường `observation.*` (đầu vào của mạng) + `action` (đáp án của thầy). Ảnh được lưu thành video mp4, đọc ra thành
tensor `[3, cao, rộng]` giá trị 0-1.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
import matplotlib.pyplot as plt

ds = LeRobotDataset('local/so101_mini', root=DS, video_backend='pyav')
print(f'{ds.num_episodes} ván, {ds.num_frames} khung, {ds.fps} khung/giây')
for name, ft in ds.features.items():
    print(f'  {name:28s} {ft["dtype"]:8s} {ft["shape"]}')

f = ds[60]                                           # khung thứ 60 của ván đầu: tay đang đi tới hộp
# Hai thứ khác đơn vị: state = 5 góc khớp tính bằng ĐỘ + độ mở kẹp (0 đóng .. 100 mở); action = 5 góc khớp đích tính bằng RAD +
# lệnh kẹp (> 0 mở, < 0 khép). Mạng không bận tâm: bộ chuẩn hoá đưa mọi số về cùng thang trước khi vào mạng.
print('state  (độ, kẹp 0-100):  ', f['observation.state'].numpy().round(1))
print('action (rad, kẹp -1..1): ', f['action'].numpy().round(3))
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
for a, cam in zip(ax, ['top', 'wrist']):
    a.imshow(f[f'observation.images.{cam}'].permute(1, 2, 0)); a.set_title(cam); a.axis('off')

## 2. Train: ACT học bắt chước thầy, 300 step (~2 phút trên GTX 1650)

`learning/train.py` đọc file YAML (một thí nghiệm = một file trong `learning/configs/`) rồi gọi `lerobot-train`. Tham số gõ thêm
ghi đè YAML: ở đây đổi dataset, chỗ lưu, và thu nhỏ số step / batch. Đọc log theo bảng "Thuật ngữ khi train" trong `docs/so_tay_ai.md`:
`loss` phải giảm dần (chạy thử: 10,9 ở step 50 -> 2,8 ở step 300; bản thật xuống ~0,1).

In [ ]:
OUT = MINI / 'outputs' / 'act_mini'
shutil.rmtree(OUT, ignore_errors=True)               # lerobot-train báo lỗi nếu thư mục đã có (trừ khi --resume)
run('sim_engine.learning.train', REPO / 'learning/configs/act_box4_topwrist.yaml',
    '--dataset.repo_id=local/so101_mini', f'--dataset.root={DS}', f'--output_dir={OUT}', '--job_name=act_mini',
    '--steps=300', '--batch_size=4', '--save_freq=150', '--log_freq=50', '--num_workers=0',
    keep=lambda line: 'step:' in line or 'Error' in line or 'lerobot-train' in line)
print(sorted(p.name for p in (OUT / 'checkpoints').iterdir()))
print(sorted(p.name for p in (OUT / 'checkpoints/last/pretrained_model').iterdir()))

## 3. Chấm: mạng tự lái trên vài seed của bộ chấm (không có thầy)

`learning/eval_policy.py`: mỗi bước mạng nhìn ảnh + góc khớp, ra action, mô phỏng làm theo, lặp lại (vòng kín). Mạng mini gần như
chắc chắn hỏng (thường "Chưa kẹp được").

In [ ]:
run('sim_engine.learning.eval_policy', '--policy', OUT / 'checkpoints/last/pretrained_model', '--seeds', '0-1')

### 3b. So với một checkpoint thật đã train ở nhà

`D:/vla/outputs/train/act_box4_topwrist` (200 ván, 20k step, ~36-40 %). Ô thứ hai gộp hai checkpoint của nó (trung bình action),
cùng cách đã đưa 3 mạng DAgger từ 77,8 % lên 84,8 %: chậm hơn vì mỗi bước chạy hai mạng.

In [ ]:
REAL = Path('D:/vla/outputs/train/act_box4_topwrist/checkpoints')
run('sim_engine.learning.eval_policy', '--policy', REAL / 'last/pretrained_model', '--seeds', '0-2')

In [ ]:
run('sim_engine.learning.eval_policy', '--policy', REAL / '010000/pretrained_model', REAL / '020000/pretrained_model',
    '--seeds', '0-2')

## 4. Mở hộp đen: tự viết vòng lặp nhìn -> đoán -> làm

Đây chính là thân hàm `run_episode` trong `learning/eval_policy.py`, rút gọn. Chú ý cột "hàng đợi": ACT đoán 50 action một lần
(`chunk_size`), làm 20 cái (`n_action_steps`) rồi mới đoán lại; giữa hai lần đoán mạng **không nhìn ảnh**.

In [ ]:
import numpy as np
from lerobot.common.control_utils import predict_action
from sim_engine.learning import sim_setup
from sim_engine.learning.eval_policy import load_policy

policy, pre, post = load_policy(REAL / 'last/pretrained_model', 'cuda')      # mạng + bộ chuẩn hoá vào / ra
cameras, env_state = sim_setup.spec_from_features(policy.config.input_features)  # mạng cần camera nào
task = sim_setup.TASKS['box'].format(color='red')
env, options = sim_setup.make_env(0, 'box', 0.04, images=True)              # seed 0, hộp 4 cm
obs, info = env.reset(seed=0, options=options)
policy.reset()

for t in range(80):
    frame = sim_setup.to_frame(env, obs, np.zeros(7), task, cameras, env_state)          # env -> khung LeRobot
    observation = {k: v for k, v in frame.items() if k.startswith('observation.')}
    if t == 0:
        print({k: getattr(v, 'shape', v) for k, v in observation.items()})
    queued = len(policy._action_queue)                                                     # 0 = sắp đoán chuỗi mới
    action = predict_action(observation, policy, torch.device('cuda'), pre, post, use_amp=False, task=task)
    obs, _, _, _, info = env.step(sim_setup.to_env_action(action.cpu().numpy()))           # 6 số -> 7 số env nhận
    if t % 10 == 0 or queued == 0:
        print(f'bước {t:3d} | hàng đợi trước khi gọi: {queued:2d} | action {action.cpu().numpy().ravel().round(2)} '
              f'| đầu kẹp cách hộp {np.linalg.norm(info["ee_pos"] - info["box_pos"]) * 100:.1f} cm')
env.close()

## Đọc code theo thứ tự nào

| Thứ tự | File | Đọc để hiểu |
|---|---|---|
| 1 | `learning/sim_setup.py` | Cầu nối env <-> LeRobot: camera nào, khung gồm gì, đổi action 6 <-> 7 số, bộ seed chấm |
| 2 | `learning/record_dataset.py` | Thầy làm mẫu, chỉ giữ ván thành công (ô 1) |
| 3 | `learning/train.py` + `learning/configs/*.yaml` | Train chỉ là chọn tham số; vòng lặp thật nằm trong LeRobot (ô 2) |
| 4 | `learning/eval_policy.py` | `run_episode` (ô 4), `act` (gộp checkpoint), bảng tóm tắt gần / xa |
| 5 | `learning/dagger.py` | AI lái, thầy tiếp quản, ghi đoạn thầy sửa |
| 6 | `learning/merge_datasets.py` | Gộp nhiều dataset trước khi train |
| 7 | `colab/so101_act_colab.ipynb` | Cùng các bước, chạy nền trên GPU lớn, lưu lên Drive |

Thuật ngữ trong log và bảng kết quả: `docs/so_tay_ai.md`, mục 7.